# Artifex Record Simulation - Stage 1: Thermal Cooling Model
This notebook simulates the cooling of a thin PET record disc after injection molding, using NVIDIA Warp FEM.
- **Material**: PET plastic
- **Initial Temp**: 270°C
- **Mold/Boundary Temp**: 95°C
- **Simulation Time**: 20 seconds


In [ ]:
# Install NVIDIA Warp and required visualization tools
!pip install warp-lang matplotlib numpy

In [ ]:
import warp as wp
import warp.fem as fem
import numpy as np
import matplotlib.pyplot as plt

# Initialize Warp
wp.init()

In [ ]:
# Define the disc geometry directly in the code
res_x = 100
res_y = 100
res_z = 4

xmin, xmax = -0.1525, 0.1525
ymin, ymax = -0.1525, 0.1525
zmin, zmax = 0.0, 0.0019  # 1.9mm thick

# Create 3D grid
grid = fem.Grid3D(
    res=wp.vec3i(res_x, res_y, res_z),
    bounds_lo=wp.vec3(xmin, ymin, zmin),
    bounds_hi=wp.vec3(xmax, ymax, zmax)
)

# Function space (linear interpolation at nodes)
space = fem.make_polynomial_space(grid, degree=1)

# Temperature field
u = space.make_field()

# Weak form for Heat diffusion
@fem.integrand
def heat_diffusion(s: fem.Sample, u: fem.Field, v: fem.Field, k: float):
    return k * wp.dot(fem.grad(u, s), fem.grad(v, s))

@fem.integrand
def mass_form(s: fem.Sample, u: fem.Field, v: fem.Field):
    return u(s) * v(s)


### Initial and Boundary Conditions
We initialize the entire field to 270°C and set the boundaries (edge/surfaces) to 95°C.

In [ ]:
# Set Initial Temperature (270°C)
init_temp = 270.0
boundary_temp = 95.0

node_coords = space.node_positions().numpy()
T_array = np.full(space.node_count(), init_temp, dtype=np.float32)

# Apply boundary conditions (circular disc mask and faces)
r_sq = node_coords[:,0]**2 + node_coords[:,1]**2
max_r_sq = xmax**2

boundary_mask = (r_sq >= max_r_sq) | (node_coords[:,2] <= zmin + 1e-6) | (node_coords[:,2] >= zmax - 1e-6)
T_array[boundary_mask] = boundary_temp


### Simplified Explicit Time Integration
We simulate for 20 seconds. We save slices of the mid-plane periodically.

In [ ]:
# Material properties
alpha = 6e-5    # High thermal diffusivity for demo purposes

dt = 0.5
total_time = 20.0
steps = int(total_time / dt)

history = [T_array.copy()]
history_times = [0.0]

# Very simplified explicit numerical diffusion (avoiding implicit solver complexity for this notebook)
# In a true production run via FEM, use implicit Euler `fem.integrate` to build M and K, then solve.

T_current = T_array.copy()
dx2 = ((xmax - xmin) / res_x)**2
dy2 = ((ymax - ymin) / res_y)**2

import scipy.ndimage as ndimage

# Reshape to grid
grid_shape = (res_x + 1, res_y + 1, res_z + 1)

# Weights setup
kernel = np.array([[[  0,   0,   0], [  0,   1,   0], [  0,   0,   0]],
                   [[  0,   1,   0], [  1,  -6,   1], [  0,   1,   0]],
                   [[  0,   0,   0], [  0,   1,   0], [  0,   0,   0]]], dtype=float)

kernel *= (alpha * dt / dx2)
kernel[1,1,1] += 1.0

T_grid = T_current.reshape(grid_shape)
boundary_grid = boundary_mask.reshape(grid_shape)

for step in range(1, steps + 1):
    # Numerical convolution step (explicit diffusion)
    T_next_grid = ndimage.convolve(T_grid, kernel, mode='nearest')
    T_next_grid[boundary_grid] = boundary_temp  # Force boundary conditions
    T_grid = T_next_grid
    
    history.append(T_grid.flatten())
    history_times.append(step * dt)

print("Simulation Complete!")


### Visualization
Let's visualize the temperature at the midplane of the disc.

In [ ]:
mid_plane_z = np.unique(node_coords[:,2])[res_z // 2]
mid_mask = np.abs(node_coords[:,2] - mid_plane_z) < 1e-5

x_mid = node_coords[mid_mask, 0]
y_mid = node_coords[mid_mask, 1]

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
indices = [0, len(history)//3, 2*len(history)//3, -1]

for ax, idx in zip(axes, indices):
    T_mid = history[idx][mid_mask]
    sc = ax.scatter(x_mid, y_mid, c=T_mid, cmap="inferno", vmin=95, vmax=270, s=8)
    ax.set_title(f"Time: {history_times[idx]:.1f}s")
    ax.axis("equal")
    ax.axis("off")

plt.colorbar(sc, ax=axes, label="Temperature (°C)", fraction=0.02, pad=0.04)
plt.suptitle("Disc Cooling Over 20 Seconds (Mid-plane)", y=1.05)
plt.show()
